# 02. 신작이 나오면 시리즈 전작이 싸지나? (A5)

> 겜누리 · 분석 A5 · 데이터: PC 스토어 가격 이력(`price_history`) + IGDB 시리즈·출시일(`pool.csv`)

## 1단계. 데이터 분석 설계

### 왜 이 분석을 하나
"신작이 나오면 전작을 할인한다"는 말은 흔하다. 사실이라면 **신작 출시 전후로 전작 구매 시점**을 안내할 수 있다. 데이터로 확인한다.

### 가설
| # | 가설 | 귀무가설 (H0) | 검정 | 왜 이 방법인가 |
|---|---|---|---|---|
| H1 | 신작 출시 **후** 전작의 평균 할인율이 **출시 전**보다 높다 | 전후 차이가 없다 | 시리즈별 전후 비교, 대응 Wilcoxon 부호순위 검정(단측) | 같은 시리즈를 전후로 비교하는 **대응(paired) 표본**이고, 표본이 작고 정규성을 보장할 수 없어 비모수 검정을 쓴다 |
| H2 | 그 증가가 **시리즈가 아닌 게임의 같은 기간 변화**보다 크다 | 일반적인 시즌 변화와 같다 | 이중차분(DiD): (전작 변화 − 대조군 변화)에 Wilcoxon | 출시 시점이 세일 시즌과 겹치면 H1만으로는 "신작 효과"와 "시즌 효과"를 구분할 수 없다. 같은 날짜의 시리즈 없는 게임을 대조군으로 쓴다 |

### 정의
- **신작**: 같은 시리즈 안에서 출시일이 가장 늦은 게임. **전작**: 신작보다 30일 이상 먼저 나온 같은 시리즈 게임.
- **할인율**: 일별 최저가 기준 할인율(%). 출시일 R을 기준으로 **전 W일 [R−W, R)** 과 **후 W일 [R, R+W)** 의 평균을 비교한다.
- 윈도우 W = 60일(30·90일로 민감도 확인).

**데이터 한계**: 이력이 약 23개월이고, 수집 대상 선정 시 **출시 180일 미만 게임은 제외**했다. 그래서 아주 최근 신작은 분석에 없다.

## 2단계. 데이터 준비

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import platform
from functools import lru_cache
from pathlib import Path
from urllib.parse import quote_plus

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

if platform.system() == "Windows":
    plt.rc("font", family="Malgun Gothic")
elif platform.system() == "Darwin":
    plt.rc("font", family="AppleGothic")
plt.rc("axes", unicode_minus=False)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

# 레포 루트 찾기 (노트북을 어디서 열어도 같은 경로를 쓴다)
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "analysis").is_dir():
    ROOT = ROOT.parent
DATA = ROOT / "analysis" / "data"
print("레포 루트:", ROOT)

In [ ]:
# 가격 이력은 DB(Replica)에서 SELECT로만 읽고, 한 번 읽으면 CSV로 저장해 다음부터 재사용한다.
# (발표 현장에서 VM·SSH 터널이 안 돼도 시연할 수 있게 하기 위함. analysis/data/ 는 Git에 올라가지 않는다.)
PRICE_CSV = Path(os.environ.get("GAMENURI_PRICE_CSV", DATA / "cache" / "price_history_pc_new.csv"))


def load_price_history(game_ids, refresh: bool = False) -> pd.DataFrame:
    """PC·새 상품(item_condition='new') 가격 이력. 컬럼: game_id, store, edition, price_krw, discount_pct, collected_at"""
    if PRICE_CSV.exists() and not refresh:
        df = pd.read_csv(PRICE_CSV, parse_dates=["collected_at"])
    else:
        from sqlalchemy import bindparam, create_engine, text
        from gamenuri_common.config import Settings

        s = Settings(_env_file=ROOT / ".env")  # 비밀번호는 출력하지 않는다
        url = (f"mysql+pymysql://{s.mariadb_user}:{quote_plus(s.mariadb_password)}"
               f"@{s.mariadb_read_host}:{s.mariadb_port}/{s.mariadb_db}")
        query = text(
            "SELECT game_id, store, edition, price_krw, regular_krw, discount_pct, collected_at "
            "FROM price_history WHERE platform = 'PC' AND item_condition = 'new' AND game_id IN :ids"
        ).bindparams(bindparam("ids", expanding=True))
        with create_engine(url).connect() as conn:  # 읽기 전용 계정, SELECT만 실행
            df = pd.read_sql(query, conn, params={"ids": [int(i) for i in game_ids]})
        PRICE_CSV.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(PRICE_CSV, index=False)
    return df[df.game_id.isin(game_ids)].copy()

In [ ]:
games = pd.read_csv(ROOT / "analysis" / "game_list" / "games_1000.csv", encoding="utf-8-sig", parse_dates=["release_date"])
pool = pd.read_csv(DATA / "game_pool" / "pool.csv", encoding="utf-8-sig", usecols=["game_id", "series_id", "game_type"])
games = games.merge(pool, on="game_id", how="left")
games["name_ko"] = games.name_ko.fillna(games.name)   # 한글 이름이 없으면 영문 이름

prices = load_price_history(games.game_id.tolist())
print(f"가격 이력: 행 {len(prices):,} · 게임 {prices.game_id.nunique():,}개 · "
      f"{prices.collected_at.min():%Y-%m-%d} ~ {prices.collected_at.max():%Y-%m-%d}")
games[["game_id", "name_ko", "series_id", "series_name", "release_date", "game_type"]].head()

In [ ]:
# 시리즈 구조 파악: 2편 이상 들어 있는 시리즈가 몇 개인가
sizes = games.dropna(subset=["series_id"]).groupby("series_id").size()
print(f"시리즈에 속한 게임 {games.series_id.notna().sum():,}개 · 시리즈 {len(sizes)}개 · 2편 이상 {int((sizes >= 2).sum())}개")
sizes[sizes >= 2].value_counts().sort_index().rename("시리즈 수").to_frame("시리즈 수").T

## 3단계. 데이터 가공
1. 게임별 **일별 최저가·할인율**을 만든다 (스토어별 마지막 가격을 이어 채움).
2. 달력 날짜 × 게임 표(`cal`)로 펼친다. 가격이 안 바뀐 날도 직전 할인율을 이어서 채운다.
3. 시리즈마다 신작(R)과 전작을 정한다.

In [ ]:
# ---- 서비스(backend/app/services/timing.py)와 같은 기준 ----
SALE_MIN_PCT = 10      # 할인율 10% 이상인 날이 이어진 구간 = 할인 1회
WINDOW_DAYS = 730      # 할인 횟수·주기 계산 기간 (최근 2년)
MIN_HISTORY_DAYS = 3   # 이력 일수가 이보다 적으면 제외


def daily_series(g: pd.DataFrame) -> pd.DataFrame:
    """게임 1개의 일별 최저가. 스토어(+에디션)별 마지막 가격을 이어 채운 뒤 그날의 최저가 한 줄만 남긴다.
    (price_history에는 가격이 바뀐 스토어의 행만 있어서 이렇게 해야 모든 스토어의 '그 시점 가격'이 된다)"""
    g = g.assign(day=g.collected_at.dt.normalize(), discount_pct=g.discount_pct.fillna(0))
    last = g.sort_values("collected_at").groupby(["day", "store", "edition"]).tail(1)
    price = last.pivot(index="day", columns=["store", "edition"], values="price_krw")
    disc = last.pivot(index="day", columns=["store", "edition"], values="discount_pct")
    price = price.sort_index().sort_index(axis=1).ffill()
    disc = disc.reindex_like(price).ffill()
    best = np.nanargmin(price.to_numpy(), axis=1)
    rows = np.arange(len(best))
    return pd.DataFrame({
        "day": price.index,
        "price_krw": price.to_numpy()[rows, best],
        "discount_pct": disc.to_numpy()[rows, best],
    })


def sale_events(d: pd.DataFrame) -> pd.DataFrame:
    """할인율이 SALE_MIN_PCT 이상인 날이 이어진 구간마다 한 건 (시작일, 최대 할인율)."""
    events, start, top = [], None, 0
    for day, pct in zip(d.day, d.discount_pct):
        if pct >= SALE_MIN_PCT:
            if start is None:
                start, top = day, pct
            else:
                top = max(top, pct)
        elif start is not None:
            events.append((start, top))
            start = None
    if start is not None:
        events.append((start, top))
    return pd.DataFrame(events, columns=["start", "max_pct"]).astype({"start": "datetime64[ns]"})

In [ ]:
END = prices.collected_at.max().normalize()
START = prices.collected_at.min().normalize()

series_by_game = {gid: daily_series(g) for gid, g in prices.groupby("game_id")}
first_day = {gid: d.day.min() for gid, d in series_by_game.items()}

cols = {}
for gid, d in series_by_game.items():
    s = d.set_index("day").discount_pct
    cols[gid] = s[~s.index.duplicated(keep="last")].reindex(pd.date_range(START, END)).ffill()
cal = pd.DataFrame(cols)          # 행: 날짜, 열: game_id, 값: 그날 최저가 기준 할인율(%)
print(f"달력 표: {cal.shape[0]}일 × {cal.shape[1]}게임")

In [ ]:
# 시리즈별 신작·전작 정하기
multi = sizes[sizes >= 2].index
pairs = []
for sid in multi:
    s = games[games.series_id == sid].sort_values("release_date")
    new = s.iloc[-1]
    prev = s[s.release_date <= new.release_date - pd.Timedelta(days=30)]
    prev = prev[prev.game_id.isin(series_by_game)]          # 가격 이력이 있는 전작만
    if len(prev) == 0 or new.game_id not in series_by_game:
        continue
    pairs.append({"series_id": sid, "series_name": new.series_name, "new_id": new.game_id, "new_name": new.name_ko,
                  "R": new.release_date, "new_type": new.game_type, "prev_ids": prev.game_id.tolist()})
pairs = pd.DataFrame(pairs)
pairs["is_remake"] = pairs.new_type.isin([8, 9])    # IGDB game_type 8=리메이크, 9=리마스터
print(f"시리즈 {len(multi)}개 → 신작·전작 쌍이 있는 시리즈 {len(pairs)}개 (신작이 리메이크·리마스터: {int(pairs.is_remake.sum())}개)")
pairs.assign(전작수=pairs.prev_ids.str.len())[["series_name", "new_name", "R", "전작수", "is_remake"]].head(10)

In [ ]:
# 시리즈가 없는(= 2편 이상 시리즈에 속하지 않는) 게임 = 대조군 후보
in_multi = set(games[games.series_id.isin(multi)].game_id)
controls = [g for g in series_by_game if g not in in_multi]
print(f"대조군 후보 {len(controls)}개 게임")

one_day = pd.Timedelta(days=1)

def mean_disc(gid, a, b, cal_=None):
    """[a, b) 구간의 평균 할인율 (cal_을 안 주면 전체 스토어 기준 cal)"""
    return (cal if cal_ is None else cal_)[gid].loc[a:b - one_day].mean()

def change(gid, R, W, cal_=None, first_=None):
    """출시일 R 전후 W일의 평균 할인율 차이(%p). 이력이 모자라면 NaN."""
    c, f = (cal if cal_ is None else cal_), (first_day if first_ is None else first_)
    if gid not in c.columns or f[gid] > R - pd.Timedelta(days=W) or R + pd.Timedelta(days=W) > END + one_day:
        return np.nan
    return mean_disc(gid, R, R + pd.Timedelta(days=W), c) - mean_disc(gid, R - pd.Timedelta(days=W), R, c)

def build(pairs, W, cal_=None, first_=None):
    rows = []
    for p in pairs.itertuples():
        prev = [change(g, p.R, W, cal_, first_) for g in p.prev_ids]
        if np.all(np.isnan(prev)):
            continue
        ctrl = [change(c, p.R, W, cal_, first_) for c in controls]
        rows.append({"series_name": p.series_name, "new_name": p.new_name, "R": p.R, "is_remake": p.is_remake,
                     "n_prev": int(np.sum(~np.isnan(prev))), "change": np.nanmean(prev),
                     "ctrl_change": np.nanmean(ctrl), "prev_ids": p.prev_ids})
    out = pd.DataFrame(rows)
    out["did"] = out.change - out.ctrl_change
    return out

W = 60
res = build(pairs, W)
print(f"전후 {W}일 이력이 모두 있는 시리즈: {len(res)}개")
res[["series_name", "new_name", "R", "n_prev", "change", "ctrl_change", "did"]].round(1).head(10)

## 4단계. 데이터 분석

### H1. 신작 출시 후 전작 할인율이 올랐나? (대응 Wilcoxon, 단측)

In [ ]:
def wil(x, alt="greater"):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    return stats.wilcoxon(x, alternative=alt) if np.any(x != 0) else None

w_h1 = wil(res.change)
print(f"시리즈 {len(res)}개 · 전작 할인율 변화 중앙값 {res.change.median():+.1f}%p · 평균 {res.change.mean():+.1f}%p")
print(f"올라간 시리즈 {(res.change > 0).sum()}개 / 내려간 시리즈 {(res.change < 0).sum()}개")
print(f"Wilcoxon(변화 > 0): 통계량 {w_h1.statistic:.0f}, p = {w_h1.pvalue:.4g}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for _, r in res.iterrows():
    before = np.nanmean([mean_disc(g, r.R - pd.Timedelta(days=W), r.R) for g in r.prev_ids])
    ax[0].plot([0, 1], [before, before + r.change], color="#4c78a8" if r.change >= 0 else "#e45756", alpha=.5, marker="o", ms=3)
ax[0].set(xticks=[0, 1], xticklabels=[f"출시 전 {W}일", f"출시 후 {W}일"], ylabel="전작 평균 할인율(%)", title="시리즈별 전작 할인율 (선 하나 = 시리즈 1개)")
ax[1].hist(res.change.dropna(), bins=20, color="#4c78a8"); ax[1].axvline(0, color="black", lw=1)
ax[1].axvline(res.change.median(), color="red", ls="--", label=f"중앙값 {res.change.median():+.1f}%p")
ax[1].set(title="전작 할인율 변화 (후 − 전)", xlabel="%p"); ax[1].legend()
plt.tight_layout(); plt.show()

### 차트 해설 — H1
- **왼쪽**: 선이 오른쪽 위로 올라가면(파랑) 출시 후 전작 할인율이 올랐다는 뜻이다. 빨강은 내려간 시리즈.
- **오른쪽**: 분포가 0보다 오른쪽으로 치우치고 중앙값이 양수면 가설을 뒷받침한다.

### H2. 시즌 효과를 걷어내도 남는가? (이중차분)
출시일이 세일 시즌과 겹치면 모든 게임의 할인율이 같이 오른다. 같은 날짜 전후로 **시리즈가 아닌 게임들의 할인율 변화(대조군 평균)** 를 빼서 순수한 신작 효과만 본다.

In [ ]:
w_h2 = wil(res.did)
print(f"전작 변화 {res.change.median():+.1f}%p, 대조군 변화 {res.ctrl_change.median():+.1f}%p(중앙값) → 이중차분 중앙값 {res.did.median():+.1f}%p")
print(f"Wilcoxon(DiD > 0): 통계량 {w_h2.statistic:.0f}, p = {w_h2.pvalue:.4g}")

# 이벤트 스터디: 신작 출시일(0일) 기준 전작 vs 대조군 평균 할인율 추이
offs = pd.timedelta_range(-pd.Timedelta(days=W), pd.Timedelta(days=W) - one_day, freq="D")
prev_curves, ctrl_curves = [], []
for r in res.itertuples():
    idx = r.R + offs
    prev_curves.append(np.nanmean([cal[g].reindex(idx).to_numpy() for g in r.prev_ids], axis=0))
    ctrl_curves.append(np.nanmean([cal[c].reindex(idx).to_numpy() for c in controls], axis=0))
x = offs.days
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(x, np.nanmean(prev_curves, axis=0), label="시리즈 전작", color="#e45756")
ax.plot(x, np.nanmean(ctrl_curves, axis=0), label="대조군(시리즈 없는 게임)", color="#4c78a8")
ax.axvline(0, color="black", ls="--"); ax.set(title="신작 출시일 전후 평균 할인율", xlabel="출시일 기준 일수", ylabel="할인율(%)"); ax.legend()
plt.show()

### 차트 해설 — H2
0일(검은 점선)은 신작 출시일이다. **빨강(전작)이 출시 후에 파랑(대조군)보다 더 올라가면** 시즌과 무관한 신작 효과가 있다는 뜻이다. 두 선이 같이 움직이면 그냥 시즌 효과다.

### 민감도 확인
윈도우 길이와 "신작이 리메이크·리마스터인 경우"를 바꿔도 결론이 같은지 본다.

In [ ]:
rows = []
for W_ in (30, 60, 90):
    r_ = build(pairs, W_)
    for label, sub in (("전체", r_), ("리메이크·리마스터 제외", r_[~r_.is_remake])):
        a, b = wil(sub.change), wil(sub.did)
        rows.append({"윈도우(일)": W_, "대상": label, "시리즈 수": len(sub),
                     "변화 중앙값(%p)": round(sub.change.median(), 1), "H1 p값": round(a.pvalue, 4) if a else np.nan,
                     "DiD 중앙값(%p)": round(sub.did.median(), 1), "H2 p값": round(b.pvalue, 4) if b else np.nan})
sens = pd.DataFrame(rows)
sens

### 민감도 2 — Steam(공식 스토어)만 사용
지금까지는 **모든 스토어의 일별 최저가**로 할인율을 계산했다. 키 판매점은 신작과 무관하게 자주 할인하기 때문에,
"퍼블리셔가 신작 때 전작을 할인한다"는 가설은 **Steam 가격만** 볼 때 더 정확할 수 있다. 같은 분석을 Steam 행만으로 반복한다.

In [ ]:
def make_calendar(prices_sub):
    sbg = {gid: daily_series(g) for gid, g in prices_sub.groupby("game_id")}
    first = {gid: d.day.min() for gid, d in sbg.items()}
    cols = {}
    for gid, d in sbg.items():
        s = d.set_index("day").discount_pct
        cols[gid] = s[~s.index.duplicated(keep="last")].reindex(pd.date_range(START, END)).ffill()
    return pd.DataFrame(cols), first

cal_steam, first_steam = make_calendar(prices[prices.store == "steam"])
print(f"Steam 행 {int((prices.store == 'steam').sum()):,}개 · 게임 {cal_steam.shape[1]}개")

res_s = build(pairs, W, cal_steam, first_steam)
w_h1_s, w_h2_s = wil(res_s.change), wil(res_s.did)
print(f"[Steam만, W={W}일] 시리즈 {len(res_s)}개 · 전작 할인율 변화 중앙값 {res_s.change.median():+.1f}%p · 평균 {res_s.change.mean():+.1f}%p")
print(f"올라간 시리즈 {(res_s.change > 0).sum()}개 / 내려간 시리즈 {(res_s.change < 0).sum()}개")
print(f"H1 Wilcoxon(변화 > 0): p = {w_h1_s.pvalue:.4g}   |   H2 이중차분 중앙값 {res_s.did.median():+.1f}%p, p = {w_h2_s.pvalue:.4g}")

rows = []
for W_ in (30, 60, 90):
    for label, c_, f_ in (("전체 스토어 최저가", None, None), ("Steam만", cal_steam, first_steam)):
        r_ = build(pairs, W_, c_, f_)
        a, b = wil(r_.change), wil(r_.did)
        rows.append({"윈도우(일)": W_, "할인율 기준": label, "시리즈 수": len(r_),
                     "변화 중앙값(%p)": round(r_.change.median(), 1), "H1 p값": round(a.pvalue, 4) if a else np.nan,
                     "DiD 중앙값(%p)": round(r_.did.median(), 1), "H2 p값": round(b.pvalue, 4) if b else np.nan})
sens_store = pd.DataFrame(rows)
sens_store

In [ ]:
# 이벤트 스터디(Steam만): 신작 출시일(0일) 전후 평균 할인율
offs = pd.timedelta_range(-pd.Timedelta(days=W), pd.Timedelta(days=W) - one_day, freq="D")
prev_c, ctrl_c = [], []
for r in res_s.itertuples():
    idx = r.R + offs
    prev_c.append(np.nanmean([cal_steam[g].reindex(idx).to_numpy() for g in r.prev_ids if g in cal_steam.columns], axis=0))
    ctrl_c.append(np.nanmean([cal_steam[c].reindex(idx).to_numpy() for c in controls if c in cal_steam.columns], axis=0))
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(offs.days, np.nanmean(prev_c, axis=0), label="시리즈 전작 (Steam)", color="#e45756")
ax.plot(offs.days, np.nanmean(ctrl_c, axis=0), label="대조군 (Steam)", color="#4c78a8")
ax.axvline(0, color="black", ls="--"); ax.set(title="신작 출시일 전후 평균 할인율 — Steam만", xlabel="출시일 기준 일수", ylabel="할인율(%)"); ax.legend()
plt.show()

### 차트 해설 — Steam만
스토어를 Steam으로 좁혀도 빨강(전작)이 출시 후 파랑(대조군)보다 더 올라가지 않으면, 결론이 "키 판매점이 섞여서"가 아니라는 뜻이다.

### 민감도 3 — 그래프의 출렁임은 우연인가? (플라시보 검정)
위 그래프에서 전작(빨강)은 출시 후 떨어졌다가 다시 오르는 것처럼 보인다. 하지만 시리즈 29개만 평균 낸 곡선은 **원래 크게 출렁인다**(시리즈 하나의 할인은 있다/없다로 갑자기 바뀌기 때문).
그래서 **신작 출시일이 아닌 임의의 날짜**를 기준으로 같은 곡선을 500번 그려, 실제 곡선이 그 우연의 범위를 벗어나는지 본다. 범위를 벗어나야 "출시일에 뭔가 일어났다"고 말할 수 있다.

- **스캔 통계량**: 출시 후 15일 이동평균이 출시 전 평균에서 벗어난 최대 폭. 어느 구간을 볼지 그래프를 보고 고르면 결과가 부풀려지므로, 모든 구간 중 가장 큰 값을 임의 날짜에서도 똑같이 구해 비교한다.

In [ ]:
rng = np.random.default_rng(7)
N_PLACEBO = 500
arr = {g: cal_steam[g].to_numpy() for g in cal_steam.columns}
dates = cal_steam.index
xs = np.arange(-W, W)

def curve_at(prev_ids, R):
    p = dates.get_loc(R)
    return np.nanmean([arr[g][p - W:p + W] for g in prev_ids if g in arr], axis=0)

def valid_dates(prev_ids):
    """그 전작들이 앞뒤 W일 이력을 모두 가진 날짜만"""
    return [d_ for d_ in dates
            if all(first_steam.get(g, pd.Timestamp.max) <= d_ - pd.Timedelta(days=W) for g in prev_ids if g in first_steam)
            and d_ + pd.Timedelta(days=W) <= END]

rows_s = list(res_s.itertuples())
obs = np.nanmean([curve_at(r.prev_ids, r.R) for r in rows_s], axis=0)
vdates = {r.series_name: valid_dates(r.prev_ids) for r in rows_s}
plc = np.empty((N_PLACEBO, 2 * W))
for k in range(N_PLACEBO):
    plc[k] = np.nanmean([curve_at(r.prev_ids, vdates[r.series_name][rng.integers(len(vdates[r.series_name]))])
                         for r in rows_s], axis=0)
lo, hi = np.percentile(plc, [2.5, 97.5], axis=0)

def scan(c, k=15):
    ma = np.convolve(c[W:], np.ones(k) / k, mode="valid")        # 출시 후 15일 이동평균
    return np.abs(ma - c[:W].mean()).max()                        # 출시 전 평균과의 최대 차이

s_obs, s_plc = scan(obs), np.array([scan(c) for c in plc])
p_scan = float((s_plc >= s_obs).mean())
out_share = float(((obs < lo) | (obs > hi)).mean())
print(f"우연 범위(95%) 밖으로 나간 날: {out_share:.1%} (순수 우연이면 약 5%)")
print(f"스캔 통계량: 관측 {s_obs:.1f}%p / 임의 날짜 중앙값 {np.median(s_plc):.1f}, 95번째 {np.percentile(s_plc, 95):.1f} → p = {p_scan:.3f}")
print(f"곡선 출렁임(표준편차): 관측 {obs.std():.1f} vs 임의 날짜 평균 {plc.std(axis=1).mean():.1f}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.fill_between(xs, lo, hi, color="lightgray", label="임의 날짜 기준 95% 범위")
ax.plot(xs, obs, color="#e45756", label="시리즈 전작 (신작 출시일 기준)")
ax.axvline(0, color="black", ls="--")
ax.set(title="실제 곡선 vs 우연의 범위 — Steam만", xlabel="출시일 기준 일수", ylabel="할인율(%)"); ax.legend()
plt.show()

### 차트 해설 — 플라시보
빨간 선이 회색 띠 **안에** 있으면 출렁임이 우연으로 설명된다는 뜻이다. 띠를 뚜렷하게 벗어나는 구간이 있어야 출시일 효과를 의심할 수 있다.

### 탐색 — 회사(퍼블리셔)별로 다른가?
회사마다 전략이 달라서(어떤 회사는 신작 때 전작을 할인하고, 어떤 회사는 멈춘다) 전체 평균에서는 상쇄됐을 수 있다. 다만 시리즈 29개가 23개 회사에 흩어져 있어서 **회사별 검정은 불가능**하다(2개 이상 가진 회사는 5곳). 그래서 세 가지를 **탐색적으로** 본다 (가설을 만들기 위한 분석이지, 확정 결론이 아니다).

1. **회사 규모**: 수집 대상 안에서 6편 이상 가진 회사(대형) vs 나머지 — Mann-Whitney
2. **평균으로의 회귀**: 출시 전에 이미 할인율이 높던 게임은 그 뒤 내려가기 쉽다 — 출시 전 할인율과 변화의 상관, 회귀
3. **이질성**: 시리즈마다 변화가 갈리는 폭이 우연(임의 날짜)보다 큰가 — 회사별 전략 차이가 있다면 퍼짐이 커야 한다

In [ ]:
import statsmodels.api as sm

pub = games.set_index("game_id").publisher
pub_n = games.publisher.value_counts()                 # 수집 대상 1,000개 안에서 회사가 가진 게임 수 = 규모의 대용치
comp = res_s.merge(pairs[["series_name", "new_id"]], on="series_name").copy()
comp["publisher"] = comp.new_id.map(pub)
comp["big"] = comp.publisher.map(pub_n) >= 6
comp["before"] = [np.nanmean([mean_disc(g, R - pd.Timedelta(days=W), R, cal_steam) for g in ids if g in cal_steam.columns])
                  for R, ids in zip(comp.R, comp.prev_ids)]
print(f"시리즈 {len(comp)}개 · 회사 {comp.publisher.nunique()}곳 · 2개 이상 가진 회사 {(comp.publisher.value_counts() >= 2).sum()}곳")

# 1) 회사 규모
display(comp.groupby("big").agg(시리즈수=("change", "size"), 변화중앙값=("change", "median"),
                                 변화평균=("change", "mean"), 출시전할인중앙값=("before", "median")).round(1)
        .rename(index={True: "대형(6편↑)", False: "소형"}))
p_big = stats.mannwhitneyu(comp[comp.big].change, comp[~comp.big].change).pvalue
print(f"[1] 대형 vs 소형 Mann-Whitney(양측): p = {p_big:.4f}")

# 2) 평균으로의 회귀
rho, p_rho = stats.spearmanr(comp.before, comp.change)
print(f"[2] 출시 전 할인율과 변화의 Spearman 상관: rho = {rho:+.2f}, p = {p_rho:.4f}")
ols = sm.OLS(comp.change, sm.add_constant(pd.DataFrame({"before": comp.before, "big": comp.big.astype(float)}))).fit()
display(ols.summary().tables[1])

# 3) 이질성: 시리즈별 변화의 표준편차가 임의 날짜보다 큰가
rng = np.random.default_rng(11)
def chg_at(ids, R):
    p = dates.get_loc(R)
    return np.nanmean([np.nanmean(arr[g][p:p + W]) - np.nanmean(arr[g][p - W:p]) for g in ids if g in arr])

vd_list = [valid_dates(ids) for ids in comp.prev_ids]
sd_plc = np.array([np.std([chg_at(ids, v[rng.integers(len(v))]) for ids, v in zip(comp.prev_ids, vd_list)], ddof=1)
                   for _ in range(N_PLACEBO)])
sd_obs = comp.change.std(ddof=1)
p_het = float((sd_plc >= sd_obs).mean())
print(f"[3] 시리즈별 변화의 표준편차: 관측 {sd_obs:.1f}%p / 임의 날짜 평균 {sd_plc.mean():.1f}, 95번째 {np.percentile(sd_plc, 95):.1f} → p = {p_het:.3f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(comp.before, comp.change, c=np.where(comp.big, "#e45756", "#4c78a8"), s=40)
for r in comp.itertuples():
    ax[0].annotate(r.series_name[:12], (r.before, r.change), fontsize=7, alpha=.7)
ax[0].axhline(0, color="black", lw=.8)
ax[0].set(title="출시 전 할인율 vs 출시 후 변화 (빨강=대형 회사)", xlabel="출시 전 60일 전작 평균 할인율(%)", ylabel="변화(%p)")
ax[1].hist(sd_plc, bins=25, color="lightgray"); ax[1].axvline(sd_obs, color="#e45756", label=f"관측 {sd_obs:.1f}%p")
ax[1].set(title="시리즈별 변화의 퍼짐: 임의 날짜 500회 vs 실제", xlabel="표준편차(%p)"); ax[1].legend()
plt.tight_layout(); plt.show()

### 차트 해설 — 회사별 탐색
- **왼쪽**: 점이 오른쪽 아래로 내려가는 모양이면 "출시 전 할인율이 높을수록 이후 더 내려간다"(평균으로의 회귀)는 뜻이다. 빨강(대형)과 파랑(소형)이 위아래로 갈리면 회사 규모 차이를 의심할 수 있다.
- **오른쪽**: 빨간 선이 회색 분포 한가운데 있으면, 시리즈마다 변화가 제각각인 정도가 우연(임의 날짜)과 같다는 뜻이다. 오른쪽 끝을 벗어나야 "회사마다 다르다"는 증거가 된다.

## 5단계. 결론 도출

In [ ]:
def verdict(p, alpha=0.05):
    return "채택 (유의)" if p < alpha else "기각 못 함 (유의하지 않음)"

result = pd.DataFrame([
    [f"H1 신작 후 전작 할인율 상승 (W={W}일)", f"변화 중앙값 {res.change.median():+.1f}%p, n={len(res)}", w_h1.pvalue, verdict(w_h1.pvalue)],
    ["H2 시즌 효과 제거 후에도 상승", f"DiD 중앙값 {res.did.median():+.1f}%p", w_h2.pvalue, verdict(w_h2.pvalue)],
    [f"H1 (Steam만, W={W}일)", f"변화 중앙값 {res_s.change.median():+.1f}%p, n={len(res_s)}", w_h1_s.pvalue, verdict(w_h1_s.pvalue)],
    ["H2 (Steam만)", f"DiD 중앙값 {res_s.did.median():+.1f}%p", w_h2_s.pvalue, verdict(w_h2_s.pvalue)],
    ["곡선 출렁임이 우연보다 큼 (Steam, 플라시보)", f"최대 이탈 {s_obs:.1f}%p vs 임의 날짜 중앙값 {np.median(s_plc):.1f}%p", p_scan, verdict(p_scan)],
    ["대형 회사 vs 소형 회사 변화 차이 (탐색)", f"중앙값 {comp[comp.big].change.median():+.1f} vs {comp[~comp.big].change.median():+.1f}%p", p_big, verdict(p_big)],
    ["회사별 전략 차이(변화의 퍼짐이 우연보다 큼, 탐색)", f"표준편차 {sd_obs:.1f}%p vs 임의 날짜 {sd_plc.mean():.1f}%p", p_het, verdict(p_het)],
], columns=["가설", "핵심 수치", "p값", "판정(α=0.05)"])
result["p값"] = result["p값"].map(lambda p: f"{p:.4g}")
result

### 해석과 인사이트
> 2026-11-26 실행 기준 · 신작·전작 쌍이 있는 시리즈 132개 중 **출시일 전후 60일 이력이 모두 있는 29개 시리즈**로 검정

| 가설 | 결과 | 의미 |
|---|---|---|
| H1 출시 후 전작 할인율 상승 | **기각 못 함** · 변화 중앙값 -0.1%p, 올라간 시리즈 13개 / 내려간 시리즈 15개 (p=0.86) | 신작 출시 전후로 전작 할인율이 달라졌다는 증거가 없다 |
| H2 시즌 효과 제거 후에도 상승 | **기각 못 함** · 이중차분 중앙값 -1.9%p (p=0.95) | 같은 기간 시리즈 없는 게임과 비교해도 차이가 없다 |

**민감도 확인 (결론이 같은가)**
- 윈도우 30·60·90일, 리메이크·리마스터 신작 제외: 모든 조합에서 p값이 0.77 이상이다.
- **Steam만** 사용(공식 스토어): 전작 할인율이 +3.4%p 올랐지만(올라간 시리즈 17개 / 내려간 10개, p=0.19) 같은 기간 대조군도 비슷하게 올라서, 시즌 효과를 빼면 -1.2%p가 되어 사라졌다(p=0.76). 키 판매점이 섞여서 효과가 흐려진 것은 아니었다.

**그래프의 출렁임과 회사별 차이 (탐색)**
- 출시 전후 곡선에서 전작 할인율이 +10~25일에 떨어졌다가 +35~50일에 다시 오르는 것처럼 보이지만, **임의 날짜로 그린 곡선의 우연 범위 안**이다. 우연 범위(95%) 밖으로 나간 날은 5.0%로 순수 우연 수준이고, 스캔 통계량도 관측 5.1%p로 임의 날짜 중앙값 7.0%p보다 작다 (p=0.81). 29개 시리즈만 평균 낸 곡선이 원래 크게 출렁이기 때문이다.
- **회사별**: 시리즈 29개가 23개 회사에 흩어져(2개 이상 가진 회사 5곳) 회사별 검정은 불가능하다. 대형 회사(수집 대상 6편 이상)는 변화 중앙값 -0.6%p, 소형은 +3.9%p로 방향은 짐작과 같지만 p=0.29로 우연 수준이다. 시리즈마다 변화가 갈리는 폭(표준편차 11.0%p)도 임의 날짜(12.6%p)와 같은 수준이라(p=0.87) **회사마다 전략이 다르다는 증거는 없다.**
- **평균으로의 회귀**: 출시 전 할인율이 높을수록 이후 변화가 작아지는 경향(회귀 계수 -0.45, p=0.016)이 보이지만, 순위 상관은 유의하지 않아(rho=-0.20, p=0.30) 방법에 따라 결과가 갈린다. 확실하지 않다.
- 이 탐색은 가설을 만들기 위한 것이다. 시리즈가 29개뿐이라 "회사별 차이가 없다"고 단정하지도 못한다.

**해석**: "신작이 나오면 전작이 싸진다"는 가설은 이 데이터에서 **지지되지 않았다.** 다만 시리즈가 29개뿐이라 작은 효과는 구분하기 어렵다. "효과가 없다"가 아니라 "있다는 증거를 찾지 못했다"로 말해야 한다.

### 서비스에 주는 제안
1. **"신작이 곧 나오니 전작을 기다려라"는 안내는 하지 않는다.** 이를 뒷받침하는 통계적 근거가 없다. 기다리게 해서 사용자가 손해를 볼 수 있다.
2. 전작 구매 판단은 시리즈 신작 여부가 아니라 **현재가와 역대 최저가 비교**(구매 타이밍 기능)와 정기 세일 시즌에 맡긴다.
3. **회사별 전략 차이는 서비스에 반영하지 않는다.** 증거가 없다. 같은 회사의 신작이 여러 번 나올 만큼 이력이 쌓이면 회사 단위로 다시 검정한다.
4. 가격 이력이 더 쌓이면 (시리즈 수가 늘고 전후 구간이 충분해지면) **같은 노트북으로 다시 검정**한다. 코드를 바꿀 필요 없이 다시 실행하면 된다.

### 한계
- 신작은 **이 수집 대상 안에서** 가장 늦게 나온 게임이다. 실제 최신작이 대상에 없을 수 있다.
- 수집 대상 선정에서 출시 180일 미만을 제외해 **아주 최근 신작은 분석에 없다.** 이력은 약 24개월이다.
- 인기작 위주 표본이라 전체 게임으로 일반화할 수 없고, 검정에 쓸 수 있었던 시리즈가 29개로 적어 검정력이 낮다.
- 회사 규모 구분(수집 대상 안에서 6편 이상)은 그래프를 본 뒤에 정한 기준이라 탐색용이다.
- 대조군은 "시리즈 없는 게임"이라 신작이 없다는 것만 보장한다. 다른 이유의 할인(퍼블리셔 자체 이벤트 등)까지 걷어내지는 못한다.